# SmartPump AI — Model Development, Cross-Validation & Evaluation

**Module:** `notebooks/03_model_training.ipynb`  
**Role:** Senior Machine Learning Engineer, AI Researcher, & Predictive Maintenance Specialist  
**Target:** Multiclass Classification of Internal Hydraulic Pump Leakage (`0: Normal`, `1: Warning`, `2: Fault`)  
**Dataset:** UCI Machine Learning Repository — *Condition Monitoring of Hydraulic Systems* (ID 447)  
**Primary Dataset:** `data/processed/stable_cycle_features.csv` (1,449 stable cycles, 255 features)  
**Secondary Dataset:** `data/processed/full_cycle_features.csv` (2,205 full cycles, 255 features)  
**Artifacts Produced:** `models/smartpump_best_model.joblib`, `models/model_metadata.json`, evaluation tables & plots  

---

## 1. Project Overview & Objectives

In this phase, we develop, cross-validate, and evaluate five supervised machine learning algorithms alongside a baseline DummyClassifier to diagnose hydraulic pump degradation:
1. **DummyClassifier (Baseline):** Empirical benchmark.
2. **Decision Tree:** Interpretable rule-based hierarchical classifier.
3. **Random Forest:** Ensemble bagging classifier with randomized feature subsampling.
4. **Support Vector Machine (SVC RBF):** Maximum-margin kernel method with feature scaling.
5. **Gaussian Naïve Bayes:** Fast generative probabilistic classifier under conditional independence.
6. **AdaBoost:** Iterative boosting algorithm with shallow decision trees.

### Core Evaluation Priorities:
- **Fault Recall (Primary):** In industrial hydraulics, an unpredicted severe internal leakage leads to cavitation, loss of actuator control, and catastrophic system shutdown. Missed faults are dangerous.
- **Fault Precision (Secondary):** False alarms trigger costly unnecessary maintenance downtime.
- **Macro F1-Score (Tertiary):** Holistic harmonic mean balanced across all three operational conditions (`Normal`, `Warning`, `Fault`).
- **Data Leakage Immunity:** Feature selection (`VarianceThreshold`) and scaling (`StandardScaler`) are encapsulated in Scikit-learn Pipelines, ensuring zero test data leakage.


## 2. Setup and Data Loading


In [1]:
import os
import sys
from pathlib import Path

# Configure headless plotting and paths
os.environ["MPLCONFIGDIR"] = os.environ.get("TEMP", "/tmp")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import preprocessing as prep
import feature_engineering as fe
import train

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
EVALUATION_DIR = RESULTS_DIR / "evaluation"
VISUALIZATIONS_DIR = RESULTS_DIR / "visualizations"

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Load primary stable feature matrix and metadata
X_stable, y_stable, meta_stable = fe.load_processed_dataset(PROCESSED_DIR, dataset_type="stable")
print(f"Loaded Primary Stable Dataset: {X_stable.shape[0]} cycles × {X_stable.shape[1]} features")
print(f"Target distribution:\n{y_stable.value_counts().sort_index()}")


Loaded Primary Stable Dataset: 1449 cycles × 255 features
Target distribution:
target
0    489
1    480
2    480
Name: count, dtype: int64


## 3. Feature and Target Verification


In [2]:
# Verify absence of identifiers, metadata, or targets in feature matrix
assert "cycle_id" not in X_stable.columns
assert "target" not in X_stable.columns
assert "stable_flag" not in X_stable.columns
assert len(X_stable.columns) == 255

# Verify zero NaN and zero infinite values
assert X_stable.isna().sum().sum() == 0
assert not np.isinf(X_stable.to_numpy()).any()

print("Feature matrix verification: 100% CLEAN (255 valid predictive sensor features, 0 metadata leakage).")


Feature matrix verification: 100% CLEAN (255 valid predictive sensor features, 0 metadata leakage).


## 4. Training, Validation and Test Preparation (70 / 15 / 15 Split)


In [3]:
# Split stable data into 70% Train, 15% Val, 15% Test
splits = prep.split_cycle_data(
    X_stable,
    y_stable,
    metadata=meta_stable,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
    stratify=True,
)

X_train, y_train = splits["X_train"], splits["y_train"]
X_val, y_val = splits["X_val"], splits["y_val"]
X_test, y_test = splits["X_test"], splits["y_test"]

print(f"Training cycles:   {len(X_train):4d} ({len(X_train)/len(X_stable)*100:.1f}%)")
print(f"Validation cycles: {len(X_val):4d} ({len(X_val)/len(X_stable)*100:.1f}%)")
print(f"Test cycles:       {len(X_test):4d} ({len(X_test)/len(X_stable)*100:.1f}%)")

# Verify mutual exclusion
assert len(set(splits["indices"]["train"]).intersection(splits["indices"]["val"])) == 0
assert len(set(splits["indices"]["train"]).intersection(splits["indices"]["test"])) == 0
assert len(set(splits["indices"]["val"]).intersection(splits["indices"]["test"])) == 0
print("Zero partition overlap confirmed.")


Training cycles:   1014 (70.0%)
Validation cycles:  217 (15.0%)
Test cycles:        218 (15.0%)
Zero partition overlap confirmed.


## 5. Baseline Model: DummyClassifier

The DummyClassifier provides a baseline by predicting classes via empirical stratified sampling.


In [4]:
models = train.build_model_pipelines(random_state=42)
dummy_pipe = models["Dummy"]
dummy_pipe.fit(X_train, y_train)
y_pred_dummy = dummy_pipe.predict(X_val)

dummy_metrics = train.evaluate_predictions(y_val, y_pred_dummy, model_name="Dummy")
print(f"Dummy Baseline Validation Accuracy: {dummy_metrics['accuracy']:.4f}")
print(f"Dummy Baseline Macro F1:            {dummy_metrics['macro_f1']:.4f}")
print(f"Dummy Baseline Fault Recall:        {dummy_metrics['fault_recall']:.4f}")


Dummy Baseline Validation Accuracy: 0.2949
Dummy Baseline Macro F1:            0.2948
Dummy Baseline Fault Recall:        0.3056


## 6. Model 1: Decision Tree Classifier

A non-parametric single tree with `class_weight='balanced'` and `VarianceThreshold`.


In [5]:
dt_pipe = models["DecisionTree"]
dt_pipe.fit(X_train, y_train)
y_pred_dt = dt_pipe.predict(X_val)

dt_metrics = train.evaluate_predictions(y_val, y_pred_dt, model_name="DecisionTree")
print(f"DecisionTree Accuracy:     {dt_metrics['accuracy']:.4f}")
print(f"DecisionTree Macro F1:     {dt_metrics['macro_f1']:.4f}")
print(f"DecisionTree Fault Recall: {dt_metrics['fault_recall']:.4f}")
print(f"DecisionTree Fault Prec:   {dt_metrics['fault_precision']:.4f}")


DecisionTree Accuracy:     0.9908
DecisionTree Macro F1:     0.9907
DecisionTree Fault Recall: 1.0000
DecisionTree Fault Prec:   0.9730


## 7. Model 2: Random Forest Classifier

Ensemble bagging with 200 trees, `class_weight='balanced'`, and `VarianceThreshold`.


In [6]:
rf_pipe = models["RandomForest"]
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_val)

rf_metrics = train.evaluate_predictions(y_val, y_pred_rf, model_name="RandomForest")
print(f"RandomForest Accuracy:     {rf_metrics['accuracy']:.4f}")
print(f"RandomForest Macro F1:     {rf_metrics['macro_f1']:.4f}")
print(f"RandomForest Fault Recall: {rf_metrics['fault_recall']:.4f}")
print(f"RandomForest Fault Prec:   {rf_metrics['fault_precision']:.4f}")


RandomForest Accuracy:     1.0000
RandomForest Macro F1:     1.0000
RandomForest Fault Recall: 1.0000
RandomForest Fault Prec:   1.0000


## 8. Model 3: Support Vector Machine (SVC RBF)

Maximum-margin separator with RBF kernel, `StandardScaler()`, and `VarianceThreshold()`.


In [7]:
svm_pipe = models["SVM"]
svm_pipe.fit(X_train, y_train)
y_pred_svm = svm_pipe.predict(X_val)

svm_metrics = train.evaluate_predictions(y_val, y_pred_svm, model_name="SVM")
print(f"SVM Accuracy:     {svm_metrics['accuracy']:.4f}")
print(f"SVM Macro F1:     {svm_metrics['macro_f1']:.4f}")
print(f"SVM Fault Recall: {svm_metrics['fault_recall']:.4f}")
print(f"SVM Fault Prec:   {svm_metrics['fault_precision']:.4f}")


SVM Accuracy:     0.9862
SVM Macro F1:     0.9861
SVM Fault Recall: 0.9861
SVM Fault Prec:   0.9726


## 9. Model 4: Gaussian Naïve Bayes

Generative probabilistic classifier with `StandardScaler` and variance smoothing.


In [8]:
gnb_pipe = models["GaussianNB"]
gnb_pipe.fit(X_train, y_train)
y_pred_gnb = gnb_pipe.predict(X_val)

gnb_metrics = train.evaluate_predictions(y_val, y_pred_gnb, model_name="GaussianNB")
print(f"GaussianNB Accuracy:     {gnb_metrics['accuracy']:.4f}")
print(f"GaussianNB Macro F1:     {gnb_metrics['macro_f1']:.4f}")
print(f"GaussianNB Fault Recall: {gnb_metrics['fault_recall']:.4f}")
print(f"GaussianNB Fault Prec:   {gnb_metrics['fault_precision']:.4f}")
print("Note: GNB assumes feature conditional independence, which is strongly violated by correlated multi-sensor aggregates.")


GaussianNB Accuracy:     0.4286
GaussianNB Macro F1:     0.3455
GaussianNB Fault Recall: 0.1667
GaussianNB Fault Prec:   0.8000
Note: GNB assumes feature conditional independence, which is strongly violated by correlated multi-sensor aggregates.


## 10. Model 5: AdaBoost Classifier

Adaptive boosting with shallow decision trees (`max_depth=2`) and learning rate `0.5`.


In [9]:
ada_pipe = models["AdaBoost"]
ada_pipe.fit(X_train, y_train)
y_pred_ada = ada_pipe.predict(X_val)

ada_metrics = train.evaluate_predictions(y_val, y_pred_ada, model_name="AdaBoost")
print(f"AdaBoost Accuracy:     {ada_metrics['accuracy']:.4f}")
print(f"AdaBoost Macro F1:     {ada_metrics['macro_f1']:.4f}")
print(f"AdaBoost Fault Recall: {ada_metrics['fault_recall']:.4f}")
print(f"AdaBoost Fault Prec:   {ada_metrics['fault_precision']:.4f}")


AdaBoost Accuracy:     1.0000
AdaBoost Macro F1:     1.0000
AdaBoost Fault Recall: 1.0000
AdaBoost Fault Prec:   1.0000


## 11. Cross-Validation (Stratified vs. Group-Aware)

We evaluate cross-validation on the training set using:
1. **5-Fold StratifiedKFold:** Standard cycle-level cross-validation.
2. **5-Fold StratifiedGroupKFold:** Grouping by the 36 contiguous experimental blocks to assess generalization to unseen physical test runs.


In [10]:
# Stratified CV
cv_stratified = train.run_cross_validation(models, X_train, y_train, n_splits=5, random_state=42)
print("\n--- 5-Fold Stratified Cross-Validation Results ---")
print(cv_stratified.to_string(index=False))

# Group-aware CV (using experimental blocks)
full_meta = pd.read_csv(PROCESSED_DIR / "full_metadata.csv")
leak_blocks_full = (full_meta["target"] != full_meta["target"].shift()).cumsum()
stable_blocks = leak_blocks_full.loc[full_meta["is_stable"]].values
train_blocks = stable_blocks[splits["indices"]["train"]]

cv_grouped = train.run_cross_validation(models, X_train, y_train, n_splits=5, groups=train_blocks, random_state=42)
print("\n--- 5-Fold Stratified Group-Aware Cross-Validation Results ---")
print(cv_grouped.to_string(index=False))



[train] Running 5-Fold Cross-Validation (StratifiedKFold) on training set (1014 cycles)...
  Dummy          | CV Macro F1: 0.3308 (±0.0250) | CV Acc: 0.3313 | Time: 6.21s
  DecisionTree   | CV Macro F1: 0.9861 (±0.0091) | CV Acc: 0.9862 | Time: 8.18s
  RandomForest   | CV Macro F1: 0.9950 (±0.0031) | CV Acc: 0.9951 | Time: 8.96s
  SVM            | CV Macro F1: 0.9792 (±0.0079) | CV Acc: 0.9793 | Time: 7.12s
  GaussianNB     | CV Macro F1: 0.3479 (±0.0396) | CV Acc: 0.4270 | Time: 0.61s
  AdaBoost       | CV Macro F1: 0.9960 (±0.0037) | CV Acc: 0.9961 | Time: 5.02s

--- 5-Fold Stratified Cross-Validation Results ---
       model  cv_macro_f1_mean  cv_macro_f1_std  cv_accuracy_mean  cv_accuracy_std  cv_time_sec
    AdaBoost          0.996049         0.003697          0.996054         0.003688         5.02
RandomForest          0.995040         0.003147          0.995069         0.003116         8.96
DecisionTree          0.986133         0.009067          0.986202         0.009025      

## 12. Hyperparameter Tuning

We execute systematic grid search across candidate models using 5-fold CV on training data.


In [11]:
tuned_models, tuning_summary = train.tune_hyperparameters(X_train, y_train, random_state=42)
print("\n--- Hyperparameter Tuning Summary ---")
print(tuning_summary.to_string(index=False))



[train] Executing Systematic Hyperparameter Grid Search (5-Fold CV)...
  RandomForest   | Best Macro F1: 0.9950 | Best Params: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 100} | Time: 9.48s
  SVM            | Best Macro F1: 0.9802 | Best Params: {'clf__C': 100.0, 'clf__gamma': 'scale'} | Time: 1.12s
  DecisionTree   | Best Macro F1: 0.9861 | Best Params: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2} | Time: 0.75s
  AdaBoost       | Best Macro F1: 0.9960 | Best Params: {'clf__estimator__max_depth': 2, 'clf__learning_rate': 0.5, 'clf__n_estimators': 100} | Time: 17.24s

--- Hyperparameter Tuning Summary ---
       model  best_macro_f1                                                                           best_params  tuning_time_sec
    AdaBoost       0.996049 {"clf__estimator__max_depth": 2, "clf__learning_rate": 0.5, "clf__n_estimators": 100}            17.24
RandomForest       0.995040       {"clf__max_depth": null

## 13. Model Comparison on Validation Set

Evaluating all candidate models on the held-out validation set (217 cycles).


In [12]:
val_comparison, detailed_reports, fitted_pipes = train.train_and_evaluate_models(
    tuned_models, X_train, y_train, X_val, y_val, split_name="Validation"
)
print("\n=== Validation Set Performance Comparison ===")
print(val_comparison[["model", "accuracy", "macro_f1", "fault_recall", "fault_precision", "warning_f1", "normal_f1"]].to_string(index=False))



[train] Training and evaluating 6 models on Validation Set (217 cycles)...
  RandomForest   | Acc: 1.0000 | Macro F1: 1.0000 | Fault Recall: 1.0000 | Fault Prec: 1.0000
  SVM            | Acc: 0.9862 | Macro F1: 0.9861 | Fault Recall: 0.9861 | Fault Prec: 0.9726
  DecisionTree   | Acc: 0.9908 | Macro F1: 0.9907 | Fault Recall: 1.0000 | Fault Prec: 0.9730
  AdaBoost       | Acc: 1.0000 | Macro F1: 1.0000 | Fault Recall: 1.0000 | Fault Prec: 1.0000
  Dummy          | Acc: 0.2949 | Macro F1: 0.2948 | Fault Recall: 0.3056 | Fault Prec: 0.2857
  GaussianNB     | Acc: 0.4286 | Macro F1: 0.3455 | Fault Recall: 0.1667 | Fault Prec: 0.8000

=== Validation Set Performance Comparison ===
       model  accuracy  macro_f1  fault_recall  fault_precision  warning_f1  normal_f1
RandomForest  1.000000  1.000000      1.000000         1.000000    1.000000   1.000000
    AdaBoost  1.000000  1.000000      1.000000         1.000000    1.000000   1.000000
DecisionTree  0.990783  0.990739      1.000000      

## 14. Final Model Selection & Decision Rationale

Model selection criteria:
1. **Fault Recall:** Absolute priority — zero missed faults.
2. **Fault Precision:** Low false alarms.
3. **Macro F1:** Balanced multiclass performance.
4. **Cross-Validation Stability:** Lowest variance across folds.
5. **Ensemble Robustness:** Non-parametric handling of high-dimensional multi-sensor inputs.

**Selected Winner:** **Random Forest** (100% Fault Recall, 100% Fault Precision, 1.0000 Macro F1, CV standard deviation $\pm 0.0031$).


In [13]:
best_model_name = "RandomForest"
final_best_pipeline = tuned_models[best_model_name]
print(f"Selected Champion Model: {best_model_name}")
print(final_best_pipeline)


Selected Champion Model: RandomForest
Pipeline(steps=[('selector', VarianceThreshold()),
                ('clf',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])


## 15. Final Held-Out Test Set Evaluation

Refitting the selected model on combined **Train + Validation data (1,231 cycles)** and evaluating on the **Held-Out Test Set (218 cycles)** exactly once.


In [14]:
# Refit on Train + Validation
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)
final_best_pipeline.fit(X_train_val, y_train_val)

# Evaluate on Held-out Test Set
y_test_pred = final_best_pipeline.predict(X_test)
final_test_metrics = train.evaluate_predictions(y_test, y_test_pred, model_name=best_model_name)

print("\n=== Final Held-Out Test Evaluation Report ===")
print(final_test_metrics["classification_report_text"])
print("Confusion Matrix:\n", final_test_metrics["confusion_matrix"])

assert final_test_metrics["fault_recall"] == 1.0, "Fault recall must be 1.0 on test set!"
assert final_test_metrics["accuracy"] == 1.0, "Test accuracy must be 1.0!"
print("Held-out test evaluation completed: ZERO classification errors.")



=== Final Held-Out Test Evaluation Report ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        74
     Warning       1.00      1.00      1.00        72
       Fault       1.00      1.00      1.00        72

    accuracy                           1.00       218
   macro avg       1.00      1.00      1.00       218
weighted avg       1.00      1.00      1.00       218

Confusion Matrix:
 [[74  0  0]
 [ 0 72  0]
 [ 0  0 72]]
Held-out test evaluation completed: ZERO classification errors.


## 16. Feature Importance & Hydraulic Interpretability

Analyzing which physical sensor features drive the classification decisions.


In [15]:
feat_imp = train.compute_feature_importance(final_best_pipeline, list(X_stable.columns), X_val, y_val, top_n=25)
print("Top 15 Most Important Features (Gini Impurity):")
print(feat_imp.sort_values("gini_importance", ascending=False)[["feature_name", "sensor", "is_priority_sensor", "gini_importance"]].head(15).to_string(index=False))

print("\nDomain Confirmation: Top features belong to SE (Hydraulic Efficiency), FS1 (Primary Volume Flow), EPS1 (Motor Power), and PS3 (Return Pressure).")


[train] Computing permutation importance on validation set (217 cycles)...
Top 15 Most Important Features (Gini Impurity):
feature_name sensor  is_priority_sensor  gini_importance
      SE_q75     SE                True         0.082550
      SE_q25     SE                True         0.081153
      SE_rms     SE                True         0.077185
     FS1_rms    FS1                True         0.062970
     FS1_q25    FS1                True         0.056019
   SE_median     SE                True         0.055657
    FS1_mean    FS1                True         0.054852
  FS1_median    FS1                True         0.050638
     FS1_q75    FS1                True         0.041099
     SE_mean     SE                True         0.040640
      SE_std     SE                True         0.036777
      SE_max     SE                True         0.014645
    SE_slope     SE                True         0.014190
    SE_range     SE                True         0.013714
  PS3_median    PS3   

## 17. Operational Sensitivity Analysis on 756 Unstable Cycles

Evaluating the stable-trained model on the 756 transient/warm-up cycles (`stable_flag == 1`).


In [16]:
unstable_eval = train.run_unstable_sensitivity_test(final_best_pipeline, processed_dir=PROCESSED_DIR)

print(f"Total Unstable Cycles: {unstable_eval['n_unstable_cycles']}")
print(f"Ground-Truth Class Counts: {unstable_eval['true_distribution']}")
print(f"Predicted Class Counts:    {unstable_eval['predicted_distribution']}")
print(f"\nUnstable Cycles Accuracy:   {unstable_eval['accuracy']:.4f}")
print(f"Unstable Cycles Macro F1:   {unstable_eval['macro_f1']:.4f}")
print(f"Unstable Cycles Fault Recall: {unstable_eval['fault_recall']:.4f}")
print(f"Unstable Cycles Confusion Matrix:\n{unstable_eval['confusion_matrix']}")
print("\nFinding: The stable-trained Random Forest model perfectly generalizes to unstable cycles (100% recall on both Warnings and Faults), demonstrating outstanding operational robustness.")


Total Unstable Cycles: 756
Ground-Truth Class Counts: {0: 732, 2: 12, 1: 12}
Predicted Class Counts:    {0: 732, 2: 12, 1: 12}

Unstable Cycles Accuracy:   1.0000
Unstable Cycles Macro F1:   1.0000
Unstable Cycles Fault Recall: 1.0000
Unstable Cycles Confusion Matrix:
[[732   0   0]
 [  0  12   0]
 [  0   0  12]]

Finding: The stable-trained Random Forest model perfectly generalizes to unstable cycles (100% recall on both Warnings and Faults), demonstrating outstanding operational robustness.


## 18. Final Conclusions & Readiness for Step 5

### Key Findings Summary:
1. **Model Champion:** Random Forest achieved top performance:
   - 5-Fold Stratified CV Macro F1: **0.9950 (±0.0031)**
   - Validation Set Macro F1: **1.0000**
   - Held-Out Test Set Macro F1: **1.0000** (Fault Recall: **100%**, Fault Precision: **100%**)
   - Unstable Cycles Macro F1: **1.0000** (Fault Recall: **100%**)
2. **Physical Interpretability:** Primary flow rate (`FS1`), hydraulic efficiency factor (`SE`), motor power (`EPS1`), and return line pressure (`PS3`) dominate decision boundaries, aligning with fluid dynamics theory.
3. **Artifacts Exported:** Model serialized to `models/smartpump_best_model.joblib` with metadata logged in `models/model_metadata.json`.
4. **Status:** **Ready for Step 5 (FastAPI Backend & Inference Service).**


In [17]:
print("=================================================================")
print("  SmartPump AI - Step 4 Machine Learning Development COMPLETE")
print("=================================================================")
print(f"Selected Model:       {best_model_name}")
print(f"Test Set Macro F1:    {final_test_metrics['macro_f1']:.4f}")
print(f"Test Set Fault Recall:{final_test_metrics['fault_recall']:.4f}")
print(f"Unstable Set Macro F1:{unstable_eval['macro_f1']:.4f}")
print("Status: READY FOR STEP 5 (API & SERVING ENGINE)")
print("=================================================================")


  SmartPump AI - Step 4 Machine Learning Development COMPLETE
Selected Model:       RandomForest
Test Set Macro F1:    1.0000
Test Set Fault Recall:1.0000
Unstable Set Macro F1:1.0000
Status: READY FOR STEP 5 (API & SERVING ENGINE)
